# Classificação

## Importação de principais bibliotecas

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.decomposition import KernelPCA
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## Abrindo e preparando arquivo para treinamento

In [3]:
df = pd.read_csv('heart_tratado.csv', sep=',', encoding='utf-8')
df2 = pd.DataFrame.copy(df)

In [4]:
df2.replace({'Sex': {'M': 0, 'F': 1}}, inplace=True)
df2.replace({'ChestPainType': {'TA': 0, 'ATA': 1, 'NAP': 2, 'ASY': 3}}, inplace=True)
df2.replace({'RestingECG': {'Normal': 0, 'ST': 1, 'LVH': 2}}, inplace=True)
df2.replace({'ExerciseAngina': {'N': 0, 'Y': 1}}, inplace=True)
df2.replace({'ST_Slope': {'Up': 0, 'Flat': 1, 'Down': 2}}, inplace=True)
df2 = df2.infer_objects(copy=False)
df2.dtypes

C:\Users\tcunha\AppData\Local\Temp\ipykernel_13788\3803149592.py:6: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df2 = df2.infer_objects(copy=False)


Age                 int64
Sex                 int64
ChestPainType       int64
RestingBP           int64
Cholesterol       float64
FastingBS           int64
RestingECG          int64
MaxHR               int64
ExerciseAngina      int64
Oldpeak           float64
ST_Slope            int64
HeartDisease        int64
dtype: object

## Criando previsores e alvo

Previsores: previsores retirados diretamente do arquivo, sem preparação prévia

In [5]:
previsores = df2.iloc[:, 0:11].values
previsores

array([[40. ,  0. ,  1. , ...,  0. ,  0. ,  0. ],
       [49. ,  1. ,  2. , ...,  0. ,  1. ,  1. ],
       [37. ,  0. ,  1. , ...,  0. ,  0. ,  0. ],
       ...,
       [57. ,  0. ,  3. , ...,  1. ,  1.2,  1. ],
       [57. ,  1. ,  1. , ...,  0. ,  0. ,  1. ],
       [38. ,  0. ,  2. , ...,  0. ,  0. ,  0. ]], shape=(917, 11))

In [6]:
alvo = df2.iloc[:, 11].values
alvo

array([0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0,
       1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1,
       1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,

Previsores_esc: Previsores escalonados com StandardScaler, pois seguem curva normal e podem utilizar esse tipo de padronização

In [7]:
previsores_esc = StandardScaler().fit_transform(previsores)
previsores_esc

array([[-1.43220634, -0.51630861, -1.34470119, ..., -0.82431012,
        -0.83150225, -1.05109458],
       [-0.47805725,  1.9368261 , -0.27058012, ..., -0.82431012,
         0.10625149,  0.59651863],
       [-1.75025603, -0.51630861, -1.34470119, ..., -0.82431012,
        -0.83150225, -1.05109458],
       ...,
       [ 0.37007527, -0.51630861,  0.80354095, ...,  1.21313565,
         0.29380223,  0.59651863],
       [ 0.37007527,  1.9368261 , -1.34470119, ..., -0.82431012,
        -0.83150225,  0.59651863],
       [-1.64423947, -0.51630861, -0.27058012, ..., -0.82431012,
        -0.83150225, -1.05109458]], shape=(917, 11))

Previsores2: Previsores usando LabelEncoder para padronizar, talvez crie uma falsa hierarquia em variáveis categóricas nominais como se fossem ordinais

In [8]:
previsores2 = df.iloc[:, 0:11].values
previsores2

array([[40, 'M', 'ATA', ..., 'N', 0.0, 'Up'],
       [49, 'F', 'NAP', ..., 'N', 1.0, 'Flat'],
       [37, 'M', 'ATA', ..., 'N', 0.0, 'Up'],
       ...,
       [57, 'M', 'ASY', ..., 'Y', 1.2, 'Flat'],
       [57, 'F', 'ATA', ..., 'N', 0.0, 'Flat'],
       [38, 'M', 'NAP', ..., 'N', 0.0, 'Up']],
      shape=(917, 11), dtype=object)

In [9]:
previsores2[:, 1] = LabelEncoder().fit_transform(previsores[:, 1])
previsores2[:, 2] = LabelEncoder().fit_transform(previsores[:, 2])
previsores2[:, 6] = LabelEncoder().fit_transform(previsores[:, 6])
previsores2[:, 8] = LabelEncoder().fit_transform(previsores[:, 8])
previsores2[:, 10] = LabelEncoder().fit_transform(previsores[:, 10])
previsores2

array([[40, 0, 1, ..., 0, 0.0, 0],
       [49, 1, 2, ..., 0, 1.0, 1],
       [37, 0, 1, ..., 0, 0.0, 0],
       ...,
       [57, 0, 3, ..., 1, 1.2, 1],
       [57, 1, 1, ..., 0, 0.0, 1],
       [38, 0, 2, ..., 0, 0.0, 0]], shape=(917, 11), dtype=object)

Previsores3: Previsores usando LabelEncoder e OneHotEncoder para criar variáveis Dummy, de forma que não crie uma falsa hierarquia entre as diferentes possibilidades de uma variável ao escalonar.

Previsores3_esc: Escalonamento na previsores3 para normalizar sem criar falsa hierarquia.

In [10]:
previsores3 = ColumnTransformer(transformers=[('OneHot', OneHotEncoder(), [1, 2, 6, 8, 10])], remainder='passthrough').fit_transform(previsores2)
previsores3

array([[1.0, 0.0, 0.0, ..., 0, 172, 0.0],
       [0.0, 1.0, 0.0, ..., 0, 156, 1.0],
       [1.0, 0.0, 0.0, ..., 0, 98, 0.0],
       ...,
       [1.0, 0.0, 0.0, ..., 0, 115, 1.2],
       [0.0, 1.0, 0.0, ..., 0, 174, 0.0],
       [1.0, 0.0, 0.0, ..., 0, 173, 0.0]], shape=(917, 20), dtype=object)

In [11]:
previsores3_esc = StandardScaler().fit_transform(previsores3)
previsores3_esc

array([[ 0.51630861, -0.51630861, -0.22981048, ..., -0.55173333,
         1.38333943, -0.83150225],
       [-1.9368261 ,  1.9368261 , -0.22981048, ..., -0.55173333,
         0.75473573,  0.10625149],
       [ 0.51630861, -0.51630861, -0.22981048, ..., -0.55173333,
        -1.52395266, -0.83150225],
       ...,
       [ 0.51630861, -0.51630861, -0.22981048, ..., -0.55173333,
        -0.85606123,  0.29380223],
       [-1.9368261 ,  1.9368261 , -0.22981048, ..., -0.55173333,
         1.46191489, -0.83150225],
       [ 0.51630861, -0.51630861, -0.22981048, ..., -0.55173333,
         1.42262716, -0.83150225]], shape=(917, 20))

## Análise dos Componentes Principais

### PCA

n_components: Representa quantos dos principais componentes serão pegos em ordem decrescente.

In [ ]:
pca = PCA(n_components=13)
previsores_pca = pca.fit_transform(previsores3_esc)
previsores_pca

explained_variance_ratio: Mostra a soma da importância dos principais componentes escolhidos.
Ex: 0.98, significa que 98% dos dados dentro do grupo de previsores escolhido são representados pelos x componentes mais importantes, sendo x o n_components definido na célula acima.

In [ ]:
pca.explained_variance_ratio_.sum()

### Kernel PCA

Útil quando não é possível traçar uma reta para dividir os dados. Quando não são linearmente separáveis.

In [ ]:
kpca = KernelPCA(n_components=4, kernel='rbf')
previsores_kernel = kpca.fit_transform(previsores2)
previsores_kernel

### LDA

Necessita, além do atributo previsor, precisa também do alvo, para ter referência. Útil quando há muitos previsores e o alvo tem muitas classes (resultados diferentes).

In [ ]:

lda = LinearDiscriminantAnalysis(n_components=1)
previsores_lda = lda.fit_transform(previsores2, alvo)
previsores_lda

## Separação de dados de treino e teste

In [ ]:
kfold = KFold(n_splits=20, shuffle=True, random_state=0)
x_treino, x_teste, y_treino, y_teste = train_test_split(previsores3, alvo, test_size=0.3, random_state=0)

## Cross Validation: Valida acurácia do modelo com base apenas em dados de treino, separando dados de teste para aferição final, não para seleção de modelo.

### Naive Bayes: Rápido, bom quando previsores não tem correlação muito forte.

In [ ]:
modelo = GaussianNB()
resultado = cross_validate(modelo, x_treino, y_treino, cv = kfold, return_train_score=True)
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean())

### SVM (Máquina de vetor de suporte): Bom para resultados lineares e não lineares, mas com uma certa correlação. Método pode ser bem visual, podendo plotar gráficos e perceber padrões nele.

In [ ]:
modelo = SVC(kernel='rbf', random_state=0, C=1)
resultado = cross_validate(modelo, x_treino, y_treino, cv=kfold, return_train_score=True)
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean())

### Regressão Logística: Útil quando uma região tem probabilidades parecidas para duas respostas diferentes. Bom para visualização dessas probabilidades, utilizados até juntamente com outros modelos.

In [ ]:
modelo = LogisticRegression(random_state=0, max_iter=600, l1_ratio=0, tol=0.0001, C=1, solver="lbfgs")
resultado = cross_validate(modelo, x_treino, y_treino, cv=kfold, return_train_score=True)
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean())

### KNN (Vizinhos mais próximos de K): Agrupa dados de treino e identifica novos dados com base à proximidade vetorial do um novo dado a um grupo ou outro.

In [ ]:
modelo = KNeighborsClassifier(n_neighbors=9, metric="minkowski", p = 1) # n_neighbors preferência a, no máximo, raiz de máximo de dados de treino
resultado = cross_validate(modelo, x_treino, y_treino, cv = kfold, return_train_score=True)
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean())

### Árvore de Decisão: Boa para esclarecer a lógica por trás da predição. Sempre trás os atributos mais importantes mais acima na árvore e liga a ele, através de ramos, os outros atributos mais importantes e assim sucessivamente até chegar nas "folhas", que seriam as saídas do modelo, as predições.

In [ ]:
modelo = DecisionTreeClassifier(criterion="gini", random_state=0, max_depth=3, min_samples_split=2, min_samples_leaf=1)
resultado = cross_validate(modelo, x_treino, y_treino, cv = kfold, return_train_score=True)
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean()) 

### Random Forest: O modelo monta diversas árvores de decisão e, se for um problema de classificação, usa a moda, se for uma regressão, usa a média dos resultados dessas árvores. Ao contrário da Árvore de Decisão, as variáveis são escolhidas aleatoriamente. Maior custo computacional, melhor resultado e menor probabilidade de overfitting, normalmente.

In [ ]:
modelo = RandomForestClassifier(n_estimators=200, criterion="gini", random_state=0, max_depth=4)
resultado = cross_validate(modelo, x_treino, y_treino, cv = kfold, return_train_score=True)
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean())

### XGBoost (Extreme Gradient Boosting): Evolução do modelo Gradient Boosting, que, por sua vez, é uma evolução do modelo Random Forest, que, por sua vez, é uma evolução da Árvore de Decisão, ou seja, é o bichão.

In [ ]:
modelo = XGBClassifier(objective="binary:logistic", n_estimators= 250, max_depth=2, learning_rate=0.05)
resultado = cross_validate(modelo, x_treino, y_treino, cv = kfold, return_train_score=True)
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean())

### LightGBM: Também baseado em GBoosting, mais leve que XGBoost. Cresce a complexidade apenas na folha onde ele vê problema, e não no nível inteiro como nos outros. Torna árvores menores, ou seja, mais rápido.

In [ ]:
modelo = LGBMClassifier(num_leaves=150, objective="binary", max_depth=4, learning_rate=0.05, max_bin=100)
resultado = cross_validate(modelo, x_treino, y_treino, cv = kfold, return_train_score=True)

In [ ]:
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean())

### CatBoost: Parecido com LightGBM, mas com menos pre-processamento. Menos ajuste de hiperparâmetros. 

Não consegui usar o CatBoost

In [ ]:
previsores4 = df.iloc[:, 0:11]
alvo4 = df.iloc[:, 11]
x_treino, x_teste, y_treino, y_teste = train_test_split(previsores4, alvo4, test_size=0.3, random_state=0)
kfold = KFold(n_splits=20, shuffle=True, random_state=0)
colunas_cat = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

In [ ]:
modelo = CatBoostClassifier(task_type='CPU', cat_features=colunas_cat, random_state=0, eval_metric="Accuracy", verbose=100)
resultado = cross_validate(modelo, x_treino, y_treino, cv = kfold, return_train_score=True)

In [ ]:
print("Acurácia de Treino (Média): ", resultado['train_score'].mean())
print("Acurácia de Validação cruzada (Média): ", resultado['test_score'].mean())

## Melhor modelo:
XGBoost com 88% de acurácia nos dados de treino calculados através de cross validation.

In [ ]:
xg = XGBClassifier(objective="binary:logistic", n_estimators= 250, max_depth=2, learning_rate=0.05)
xg.fit(x_treino, y_treino)
previsoes_xg = xg.predict(x_teste)
print("Acurácia %.2f%%" % (accuracy_score(y_teste, previsoes_xg) * 100))

Com dados de teste, ficou em 86,59% de acurácia

## Deploy

In [12]:
np.savetxt("previsores.csv", previsores, delimiter=',')
np.savetxt("alvo.csv", alvo, delimiter=',')

Segue resto em "simulador_classificacao.ipynb"